# 02 · Modelling: 9 classifiers, walk-forward validation 2005–2020

This notebook is the brief's documented modelling deliverable:

1. **Pipeline:** the modelling table (14 features + label) and the walk-forward folds
2. **Models:** the 9 scikit-learn / XGBoost classifiers, their preparation and search spaces
3. **Training and tuning:** for each year 2005–2020, tune on 5 inner folds by macro-F1, retrain, predict that year
4. **Comparison:** macro-F1, balanced accuracy and accuracy, overall and per year, with confidence intervals
5. **Tuned settings:** which settings won in each year
6. **Shortlist:** the top 3 by macro-F1 go on to the money test in notebook 03
7. **Interpretation:** confusion matrices, coefficients and permutation importance

The test years 2021 → latest are **not used here**. They are opened once, in notebook 03, after the model is chosen.

In [ ]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.inspection import permutation_importance

from src.data import load_spy
from src.features import build_features, FEATURES, FEATURE_INFO
from src.labels import make_labels, CLASS_NAMES, CLASS_ORDER
from src.splits import walk_forward_folds, inner_splits
from src.models import (MODEL_NAMES, MODEL_SPECS, run_walk_forward, always_buy_proba, classification_scores,
                        block_bootstrap_ci, macro_f1, search_space_text, make_search)
from src import plotting as P

P.setup()
import os
QUICK = os.environ.get("SPY_QUICK") == "1"  # developer smoke test: only the last 2 validation years
RES = ROOT / ("results_quick" if QUICK else "results"); RES.mkdir(exist_ok=True)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
pd.set_option("display.max_colwidth", 120)

## 1. Modelling table and walk-forward folds

In [ ]:
px = load_spy()
feats = build_features(px)
y_all = make_labels(px["close"])
data = feats.join(y_all).dropna()
X, y = data[FEATURES], data["label"].astype(int)
print(f"{len(X):,} rows × {X.shape[1]} features, {X.index[0].date()} to {X.index[-1].date()}")

folds = walk_forward_folds(X.index)
fold_table = pd.DataFrame([f.describe(X.index) for f in folds]).set_index("fold")
fold_table

Each fold trains on every year before the predicted year, minus the last **5 rows** (their 5-day labels reach into the predicted year). Inside each training window, the 5 most recent ~1-year blocks act as practice exams for tuning. Example for fold 1:

In [ ]:
f1_ = folds[0]
d_tr = X.index[f1_.train_idx]
rows = []
for k, (tr, va) in enumerate(inner_splits(len(f1_.train_idx)), 1):
    rows.append({"inner fold": k, "inner train": f"{d_tr[tr[0]].date()} to {d_tr[tr[-1]].date()}",
                 "gap rows": va[0] - tr[-1] - 1, "inner validate": f"{d_tr[va[0]].date()} to {d_tr[va[-1]].date()}"})
pd.DataFrame(rows).set_index("inner fold")

## 2. The models

All 9 table-based classification models from Module 3. The text and image models (DistilBERT, CNN, ResNet18) don't apply to price data. Every model is a Pipeline, so scaling and oversampling are fitted on training rows only.

In [ ]:
spec_table = pd.DataFrame([search_space_text(m) for m in MODEL_NAMES]).set_index("model")
spec_table

## 3. Train, tune and predict 2005–2020

For each model and each year: draw candidate settings, score each by **macro-F1** on the 5 inner folds, keep the best mean, retrain on all training years, then predict the year. Results are cached in `results/`, so re-running the notebook is fast. Delete `results/val_*` to recompute; the first run takes roughly an hour on a laptop.

In [ ]:
val_folds = [f for f in folds if f.phase == "validate"]
if QUICK:
    val_folds = val_folds[-2:]
slug = lambda m: m.lower().replace(" ", "_").replace("-", "_")
val_proba, val_logs = {}, []
for m in MODEL_NAMES:
    pf, lf = RES / f"val_proba_{slug(m)}.parquet", RES / f"val_log_{slug(m)}.json"
    if pf.exists() and lf.exists():
        val_proba[m] = pd.read_parquet(pf)
        val_logs.append(pd.read_json(lf))
        print(f"loaded {m}")
        continue
    print(f"running {m} ...")
    p, log = run_walk_forward(m, X, y, val_folds, verbose=True)
    p.to_parquet(pf); log.to_json(lf, indent=1)
    val_proba[m] = p; val_logs.append(log)
val_idx = np.concatenate([f.test_idx for f in val_folds])
val_proba["Always predict Buy"] = always_buy_proba(X.index[val_idx], X.index[val_idx].year)
val_log = pd.concat(val_logs, ignore_index=True)

## 4. Comparison over 2005–2020

In [ ]:
y_val = y.iloc[val_idx]
order = ["Dummy", "Always predict Buy"] + [m for m in MODEL_NAMES if m != "Dummy"]
rows = []
for m in order:
    pred = val_proba[m]["pred"].reindex(y_val.index)
    s = classification_scores(y_val, pred)
    lo, hi = block_bootstrap_ci(y_val.values, pred.values, macro_f1)
    alo, ahi = block_bootstrap_ci(y_val.values, pred.values, lambda a, b: (a == b).mean())
    rows.append({"model": m, **s, "macro_f1 95% CI": f"{lo:.3f} to {hi:.3f}", "accuracy 95% CI": f"{alo:.3f} to {ahi:.3f}"})
comparison = pd.DataFrame(rows).set_index("model").sort_values("macro_f1", ascending=False)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
cmp_ = comparison.sort_values("macro_f1")
cols = [P.MODEL_COLORS.get(m, P.MUTED) for m in cmp_.index]
ax.barh(cmp_.index, cmp_["macro_f1"], color=cols, height=0.6)
for i, (m, v) in enumerate(cmp_["macro_f1"].items()):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center", color=P.INK2, fontsize=9)
base = comparison.loc["Dummy", "macro_f1"]
ax.axvline(base, color=P.INK2, ls="--", lw=1)
ax.text(base, len(cmp_) - 0.4, " dummy", color=P.INK2, fontsize=8, va="bottom")
ax.set_xlim(0, max(0.45, cmp_["macro_f1"].max() + 0.05)); ax.grid(axis="y", visible=False)
ax.set_title("Macro-F1 on all 2005–2020 predictions (higher is better)")
fig.tight_layout(); P.save(fig, "02_model_comparison")

### Macro-F1 per year

In [ ]:
per_year = pd.DataFrame({m: [macro_f1(y_val[y_val.index.year == yr], val_proba[m]["pred"].reindex(y_val.index)[y_val.index.year == yr])
                             for yr in sorted(set(y_val.index.year))] for m in comparison.index},
                        index=sorted(set(y_val.index.year))).T
fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(per_year.values, cmap=P.SEQUENTIAL, aspect="auto", vmin=0.15, vmax=0.5)
ax.set_xticks(range(per_year.shape[1]), per_year.columns, fontsize=8)
ax.set_yticks(range(per_year.shape[0]), per_year.index, fontsize=9)
for i in range(per_year.shape[0]):
    for j in range(per_year.shape[1]):
        v = per_year.iat[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if v > 0.33 else P.INK)
ax.grid(False); ax.set_title("Macro-F1 by predicted year (models sorted by overall macro-F1)")
fig.colorbar(im, ax=ax, fraction=0.02)
fig.tight_layout(); P.save(fig, "02_macro_f1_by_year")
per_year.T.describe().T[["mean", "std", "min", "max"]].sort_values("mean", ascending=False)

## 5. Tuned settings

The best settings are re-chosen every year. The table shows each year's winner and its inner-fold macro-F1 (mean ± SD over the 5 practice years). Settings that jump around from year to year mean the model has no stable best configuration.

In [ ]:
def settings_table(m):
    t = val_log[val_log.model == m][["year", "best_params", "inner_f1_mean", "inner_f1_sd"]].copy()
    params = pd.json_normalize(t["best_params"].tolist()).set_index(t.index)
    return pd.concat([t[["year"]], params, t[["inner_f1_mean", "inner_f1_sd"]]], axis=1).set_index("year")

settings = {m: settings_table(m) for m in MODEL_NAMES}
summary_rows = []
for m, t in settings.items():
    for p in [c for c in t.columns if not c.startswith("inner_")]:
        vc = t[p].astype(str).value_counts()
        changes = int((t[p].astype(str) != t[p].astype(str).shift()).sum() - 1)
        summary_rows.append({"model": m, "setting": p, "most often best": vc.index[0], "times (of 16)": int(vc.iloc[0]),
                             "best for 2020": t[p].iloc[-1], "changed between years": changes})
settings_summary = pd.DataFrame(summary_rows).set_index(["model", "setting"])
settings_summary

In [ ]:
for m in MODEL_NAMES:
    print(f"\n{m}")
    display(settings[m])

## 6. Shortlist

The top 3 models by macro-F1 over all 2005–2020 predictions, excluding the two baselines, go on to the money test in notebook 03. There, the final model and its Sell cut-off are picked by excess return, yearly win rate and Sharpe.

In [ ]:
ranked = comparison.drop(index=["Dummy", "Always predict Buy"]).sort_values("macro_f1", ascending=False)
shortlist = list(ranked.index[:3])
print("Shortlist:", shortlist)
json.dump({"shortlist": shortlist, "macro_f1": {m: float(ranked.loc[m, "macro_f1"]) for m in shortlist}},
          open(RES / "shortlist.json", "w"), indent=1)

## 7. Interpretation

### Confusion matrices (2005–2020, rows = true class, normalised by row)

In [ ]:
names = [CLASS_NAMES[c] for c in CLASS_ORDER]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, m in zip(axes, shortlist):
    pred = val_proba[m]["pred"].reindex(y_val.index)
    cm = confusion_matrix(y_val, pred, labels=CLASS_ORDER, normalize="true")
    ax.imshow(cm, cmap=P.SEQUENTIAL, vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]:.0%}", ha="center", va="center", color="white" if cm[i, j] > 0.55 else P.INK)
    ax.set_xticks(range(3), names); ax.set_yticks(range(3), names)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.grid(False); ax.set_title(m, fontsize=11)
fig.tight_layout(); P.save(fig, "02_confusion_shortlist")

In [ ]:
for m in shortlist:
    pred = val_proba[m]["pred"].reindex(y_val.index)
    print(f"\n{m}")
    print(classification_report(y_val, pred, labels=CLASS_ORDER, target_names=names, digits=3, zero_division=0))

### Which features the models relied on

**Permutation importance:** shuffle one feature and measure how much macro-F1 drops. A bigger drop means the model relied on it more. For each shortlisted model, the model is refitted with its tuned settings for each of the last 4 validation years (2017–2020). Importance is measured on that year's predictions only, never on training rows, then averaged.

In [ ]:
perm_rows = []
for m in shortlist:
    for f in [f for f in val_folds if f.year >= 2017]:
        params = val_log[(val_log.model == m) & (val_log.year == f.year)]["best_params"].iloc[0]
        est = make_search(m, len(f.train_idx)).estimator.set_params(**{f"model__{k}": (tuple(v) if isinstance(v, list) else v) for k, v in params.items()})
        est.fit(X.iloc[f.train_idx], y.iloc[f.train_idx])
        r = permutation_importance(est, X.iloc[f.test_idx], y.iloc[f.test_idx], scoring="f1_macro", n_repeats=10, random_state=0, n_jobs=1)
        perm_rows.append(pd.Series(r.importances_mean, index=FEATURES, name=(m, f.year)))
perm = pd.DataFrame(perm_rows)
perm_mean = perm.groupby(level=0).mean().T
perm_mean.index = [FEATURE_INFO[c][0] for c in perm_mean.index]
perm_mean = perm_mean[shortlist].sort_values(shortlist[0])
fig, ax = plt.subplots(figsize=(10, 5.5))
h = 0.26
for k, m in enumerate(shortlist):
    ax.barh(np.arange(len(perm_mean)) + (k - 1) * h, perm_mean[m], height=h, color=P.MODEL_COLORS[m], label=m)
ax.set_yticks(range(len(perm_mean)), perm_mean.index)
ax.axvline(0, color=P.AXIS, lw=1); ax.grid(axis="y", visible=False)
ax.legend(loc="lower right"); ax.set_xlabel("drop in macro-F1 when the feature is shuffled")
ax.set_title("Permutation importance, averaged over 2017–2020")
fig.tight_layout(); P.save(fig, "02_permutation_importance")
perm_mean.sort_values(shortlist[0], ascending=False)

### Logistic regression coefficients

Logistic regression is the most readable model: each standardised feature gets one weight per class. A positive weight pushes towards that class. These are from the model fitted for 2020 (trained 1994–2019).

In [ ]:
f_last = val_folds[-1]
params = val_log[(val_log.model == "Logistic regression") & (val_log.year == f_last.year)]["best_params"].iloc[0]
lr = make_search("Logistic regression", len(f_last.train_idx)).estimator.set_params(**{f"model__{k}": v for k, v in params.items()})
lr.fit(X.iloc[f_last.train_idx], y.iloc[f_last.train_idx])
coef = pd.DataFrame(lr[-1].coef_.T, index=[FEATURE_INFO[c][0] for c in FEATURES], columns=[CLASS_NAMES[c] for c in lr[-1].classes_])
coef.sort_values("Buy").style.background_gradient(cmap=P.DIVERGING, vmin=-coef.abs().max().max(), vmax=coef.abs().max().max()).format("{:+.3f}")

### How each year's winning setting was chosen

The search ranks every candidate by its **mean** macro-F1 over the 5 practice years and keeps the top one; on an exact tie, the first one drawn wins. Below: the candidates for the 2020 fold of each shortlisted model, and how far the winner led the runner-up in every year. All candidates are saved in `results/tuning_candidates.json`; they were recorded by re-running the seeded searches, which reproduced all 150 yearly winners exactly.

In [ ]:
CAND = json.load(open(RES / "tuning_candidates.json")) if (RES / "tuning_candidates.json").exists() else {}
if CAND:
    lead = pd.DataFrame({m: {int(y): c["candidates"][0]["mean"] - (c["candidates"][1]["mean"] if len(c["candidates"]) > 1 else np.nan)
                             for y, c in CAND[m].items() if int(y) <= 2020} for m in MODEL_NAMES if m != "Dummy"}).T
    print("Lead of the winner over the runner-up (macro-F1), per year:")
    display(lead.style.format("{:.4f}"))
    for m in shortlist:
        c = CAND[m]["2020"]
        t = pd.DataFrame([{"rank": r["rank"], "settings": r["params"], **dict(zip(c["inner_years"], r["folds"])), "mean": r["mean"], "sd": r["sd"]}
                          for r in c["candidates"]]).set_index("rank")
        print(f"
{m}: candidates for the 2020 fold (practice years {c['inner_years'][0]}-{c['inner_years'][-1]})")
        display(t.head(8))

### Indicator importance for all 9 models, every validation year

The same permutation test, now for **every model** and **all 16 validation years**. It is also run on indicator **families** shuffled together (returns, trend, momentum, Bollinger, volatility, volume), so correlated indicators can't hide behind each other. Results are cached in `results/importance.json` (about 5 minutes to compute).

In [ ]:
from src.importance import permutation_by_year, builtin_importance, GROUPS
imp_path = RES / "importance.json"
if imp_path.exists() and not QUICK:
    IMP = json.load(open(imp_path))
else:
    IMP = {}
    for m in MODEL_NAMES:
        s_, g_, est_ = permutation_by_year(m, X, y, val_folds, val_log)
        kind, bi = builtin_importance(m, est_)
        IMP[m] = {"single": s_.to_dict(orient="index"), "group": g_.to_dict(orient="index"), "builtin_kind": kind,
                  "builtin": None if bi is None else bi.to_dict(orient="index")}
    IMP["_corr"] = X.corr(method="spearman").to_dict(orient="index")
    json.dump(IMP, open(imp_path, "w"))
models_imp = [m for m in comparison.index if m in IMP and m != "Dummy"]
grp = pd.DataFrame({m: pd.DataFrame(IMP[m]["group"]).T.mean() for m in models_imp}).T
lim = np.abs(grp.values).max()
fig, ax = plt.subplots(figsize=(10, 4.6))
im = ax.imshow(grp.values, cmap=P.DIVERGING, vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(grp.shape[1]), grp.columns, rotation=20, ha="right"); ax.set_yticks(range(grp.shape[0]), grp.index)
for i in range(grp.shape[0]):
    for j in range(grp.shape[1]):
        ax.text(j, i, f"{grp.iat[i, j]:+.3f}", ha="center", va="center", fontsize=8.5, color="white" if abs(grp.iat[i, j]) > lim * 0.6 else P.INK)
ax.grid(False); ax.set_title("Drop in macro-F1 when an indicator family is shuffled, average 2005–2020 (blue = relied on)")
fig.colorbar(im, ax=ax, fraction=0.03)
fig.tight_layout(); P.save(fig, "02_group_importance_all_models")
grp.style.format("{:+.3f}")

## Modelling findings

1. **Every model beats the dummy; all but the perceptron (0.357) do so clearly.** The dummy scores 0.340 macro-F1. The top three, MLP (0.423), random forest (0.421) and logistic regression (0.421), are tied: their 95% intervals (about 0.40 to 0.44) overlap almost completely. Accuracy is 42–48% for the seven stronger models, against 35% for the dummy. That is above the brief's "35–40% is normal" because Hold is the largest class in 2005–2020 (43%) and the models lean towards it. It is far from the ~80% that would signal leakage, and the look-ahead tests pass.
2. **The skill is mostly in spotting calm weeks, not direction.** Hold recall is 0.68–0.76, while Buy and Sell recall are only 0.25–0.32. This matches the EDA: volatility tells you how big the next move will be, not which way.
3. **No model is reliable every year.** Yearly macro-F1 swings between about 0.25 and 0.46, and every model has years below the dummy (2005 and 2008 for most).
4. **Simple models did as well as complex ones.** L1-regularised logistic regression (L1 chosen in 13 of 16 years) matches the random forest and MLP, and beats XGBoost (0.399) and HistGradientBoosting (0.391). Extra flexibility found no extra signal.
5. **Unstable settings mean a flat landscape.** For the boosting models, no setting value won more than 5 of 16 years: many settings score about the same. Logistic regression, random forest (max_depth 8 in 11 years) and MLP (two layers, 32 and 16 neurons, in 14 years) were more stable.
6. **Feature effects are small; volatility and momentum matter most.** Averaged over all 16 years, every model relies on the volatility family (20-day volatility, ATR), and most rely on momentum (RSI, MACD). The trend family (moving-average ratios) slightly *hurts* 6 of the 8 models on new years. In 2017–2020, shuffling any single feature changes macro-F1 by at most about 0.04. Volatility (ATR) helps all three models, and logistic regression also leans on RSI. Several features have *negative* importance there: patterns learned from earlier years did not hold in those years. Correlated features also share credit, so single-feature importance understates groups.
7. **Shortlist for the money test:** MLP, random forest, logistic regression.